# 🌏 CSUSB Study Abroad Chatbot

## **Purpose** ⬅️
In this notebook, we’ll explore how a Python-based chatbot is created using Langchain, a powerful tool for building language-based applications. The chatbot is designed to assist students interested in studying abroad at California State University, San Bernardino (CSU San Bernardino), by answering questions about available programs, application processes, deadlines, and more.

Langchain makes it easier to connect various tools and services, allowing our chatbot to understand user questions and provide helpful responses. By the end of this notebook, you’ll have a clear understanding of how this chatbot works and how we can use Python and Langchain to build smart, helpful applications.

### **Why Learn This?** 🧠
- **Learn how chatbots work:** Understand the basics of creating a chatbot using Python and Langchain.
- **Explore study abroad options:** Discover how technology can help students find information about studying abroad at CSU San Bernardino.
- **Gain coding insights:** See how Python code can be used to build practical, real-world applications.

### **What You'll Need:** ✍️
- An installation of Python
- An installation of Pip, Python's package manager

<table>
  <tr>
    <th><h3><b>Table of Contents</b></h3></th> <th><h3><b>Flowchart Diagram</b></h3></th>
  </tr>
  <tr>
    <td><ol>
      <li><a href="#scrollTo=L7BYs7ks2_9R">Introduction and Prerequisites</a></li>
      <li><a href="#scrollTo=d4fe6881">Importing and Explaining Required Libraries</a></li>
      <li><a href="#scrollTo=b8835338">Constants and Initial Setup</a></li>
      <li><a href="#scrollTo=fcOmStXnO3bS">Downloading and Storing FAISS Index</a></li>
      <li><a href="#scrollTo=C-ylkmMDriYi">SYSTEM_PROMPT</a></li>
      <li><a href="#scrollTo=ZAGx-KiIQkGV">Setting Up the Embedding Model and Session State</a></li>
      <li><a href="#scrollTo=RjroYqd10ZUr">Token Estimation, Random Question Generation, and Input Truncation</a></li>
      <li><a href="#scrollTo=DFWKV1wLyiS_">Session Reset and Cooldown Logi</a></li>
      <li><a href="#scrollTo=iRBbQlFW6KCR">Main Page and Session Management</a></li>
    </ol></td>
    <td>
      <img src="https://raw.githubusercontent.com/DrAlzahraniProjects/csusb_spring2025_cse6550_team2/refs/heads/main/flowchart.svg">
    </td>
  </tr>
</table>


##**1. Introduction and Prerequisites**##

  ## **🤔What are we Building?**
This tutorial will guide you in creating a CSUSB Study Abroad Chatbot that helps students get
information on study-abroad programs, scholarships, visa guidance, and cultural adaptation. The
chatbot will integrate LLMs (Llama-3.1-8b), FAISS vector search, and AI-powered ranking to provide
accurate responses.
  ## ✨ Key Features :
  *  ✅ FAISS Vector Search: Stores and finds answers quickly.
  * ✅ LLMs (Llama-3.1-8b): Generates natural and accurate responses.
  * ✅ Streamlit UI: Provides a user-friendly chat interface.

  ## 🛠 Prerequisites
* ✔ Python 3.8+ installed.
* ✔ Basic understanding of AI and NLP.
* ✔ Jupyter Notebook or Streamlit setup

## **2. Installing Required Libraries** 🔧
###**Why Install These Libraries?** 🤷

Before we start building our Retrieval-Augmented Generation (RAG) chatbot, we need to install several Python libraries.
These libraries provide the tools necessary for:

* Language processing

* Embedding text for retrieval

* Connecting to vector databases

* Interacting with AI models

* Efficient indexing and searching


**Required Libraries**
Breakdown of Libraries

# Required Libraries

Below is a breakdown of the active imports in your code and their primary purposes:

| Library / Module                                 | Purpose                                                                                       |
|:-------------------------------------------------|:----------------------------------------------------------------------------------------------|
| `apscheduler.schedulers.background`              | `BackgroundScheduler` for scheduling and running periodic or one‑off background jobs          |
| `flashrank`                                      | `Ranker` & `RerankRequest` for fine‑tuned passage reranking                                   |
| `langchain_community.embeddings`                 | `FastEmbedEmbeddings` – lightweight sentence‑transformer embeddings for semantic search       |
| `langchain_community.vectorstores`               | `FAISS` vectorstore integration for efficient similarity search                               |
| `langchain_groq`                                 | `ChatGroq` LLM interface for Groq’s inference‑optimized chat models                           |
| `urllib.parse`                                   | `urlparse` for parsing and extracting components from URLs                                    |
| `os`                                             | File system operations (paths, environment variables)                                         |
| `re`                                             | Regular expressions for text processing and pattern matching                                  |
| `requests`                                       | HTTP client for making REST/API calls                                                         |
| `scrapy`                                         | Full‑featured web crawling and scraping framework                                             |
| `streamlit`                                      | High‑level API for building interactive web apps                                             |
| `streamlit.components.v1`                        | Embedding and communicating with custom Streamlit components                                  |
| `subprocess`                                     | Launching and managing external shell commands/processes                                      |
| `time`                                           | Time‑related functions (sleeps, timestamps, delays)                                           |
| `hashlib`                                        | Hashing utilities (e.g., MD5/SHA) for deduplication, caching keys                             |
| `json`                                           | JSON serialization/deserialization                                                            |
| `cachetools`                                     | `TTLCache` for in‑memory caching with time‑to‑live                                            |
| `math`                                           | Mathematical functions (ceil, floor, log, etc.)                                               |
| `typing`                                         | `Tuple`, `Dict`, `Any` for type hinting and clearer function signatures                      |


In [ ]:
!pip install fastembed flashrank langchain-groq langchain-community faiss-cpu

from flashrank import Ranker, RerankRequest
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
import os
import re
import subprocess
import time
import json

## **3. Constants and Initial Setup ⚙️**

## What Are Constants and Why Do We Need Them? 🤔
Constants are variables with fixed values that do not change throughout the execution of the program. In programming, constants help make the code more readable, maintainable, and organized. By defining constants, we give meaningful names to values that remain constant, making the code easier to modify, debug, and understand.

In this project, we define several key constants that govern the chatbot's behavior, performance, limits, and debugging settings. By using constants, we create a flexible and organized structure for managing the chatbot's operations.


## **Key Functions Using These Constants**##
##Cooldown Management ⏸️
Constants like COOLDOWN_CHECK_PERIOD, MAX_MESSAGES_BEFORE_COOLDOWN, and COOLDOWN_DURATION are used to manage the cooldown logic. They ensure that the chatbot does not get overwhelmed by too many messages in a short period of time.

* Functionality:
After a certain number of messages, the chatbot pauses to avoid exceeding token limits and gives the system time to process.

##Response Time Management ⏱️
MAX_RESPONSE_TIME controls how long the chatbot is allowed to take to generate a response. If the response time exceeds this limit, the chatbot can either retry or give a timeout error.

* Functionality:
 This constant ensures efficient performance, preventing the chatbot from taking too long to respond.

## Message Handling 📥
MAX_AI_INPUT_CHARACTERS and MAX_HISTORY_TO_USE manage the input size the AI model can handle. By limiting the number of previous messages used for context, the system remains responsive and avoids overwhelming the model with excessive data.

* Functionality: This ensures that the chatbot only uses relevant messages to generate new responses, improving efficiency.

## Debugging 🛠️
The DEBUG_MODE constant enables or disables debugging output. When enabled, the chatbot provides additional details to help with troubleshooting.

* Functionality: This is especially useful during development and testing to spot issues quickly.

## Question Generation 🎲
RANDOM_QUESTIONS_COUNT specifies how many random questions the chatbot should generate during its interaction. This can be useful for ensuring variety in conversation or for testing the system.

* Functionality: It helps keep the conversation engaging and varied, providing dynamic interactions.

###**Defining Key Constants**
Before diving into the functionality of the chatbot, let's define the constants used throughout the code. Constants are variables whose values remain unchanged and help make the code more readable and maintainable.

Here are the key constants defined in this project:

| **Constant**                       | **Description**                                                                 |
|------------------------------------|---------------------------------------------------------------------------------|
| `COOLDOWN_CHECK_PERIOD` (60.0)     | The time (in seconds) the chatbot waits between checking if it should pause.    |
| `MAX_MESSAGES_BEFORE_COOLDOWN` (10) | The chatbot allows up to 10 messages before entering a cooldown period.          |
| `COOLDOWN_DURATION` (180.0)        | The duration (in seconds) the chatbot waits during the cooldown period (3 minutes). |
| `MAX_RESPONSE_TIME` (3.0)          | The maximum time (in seconds) the chatbot is allowed to take to generate a response. |
| `ANSWER_TYPE_MAX_CHARACTERS_TO_CHECK` (30) | Checks the first 30 characters of an answer to decide the type of response.   |
| `MAX_QUESTIONS_TO_ASK` (tuple[int | None, int | None]) | Defines how many questions the chatbot should ask. If `None`, no limit. |
| `MAX_AI_INPUT_CHARACTERS` (5000)   | The maximum number of characters the chatbot can handle in one input.            |
| `MAX_HISTORY_TO_USE` (8)           | The chatbot considers the last 8 messages when generating a new response.        |
| `DEBUG_MODE` (False)               | When set to `True`, this helps with troubleshooting by showing extra information. |
| `RANDOM_QUESTIONS_COUNT` (5)       | The chatbot will generate 5 random questions during its process.                  |





In [ ]:
# Constants
COOLDOWN_CHECK_PERIOD = 60.0
MAX_MESSAGES_BEFORE_COOLDOWN = 10
COOLDOWN_DURATION = 180.0
MAX_RESPONSE_TIME = 3.0
ANSWER_TYPE_MAX_CHARACTERS_TO_CHECK = 30
MAX_AI_INPUT_CHARACTERS: int = 5000
MAX_HISTORY_TO_USE: int = 8
DEBUG_MODE: bool = False
SEGMENT_SIZE: int = 512

##**4. Downloading and Storing FAISS Index**
This code is used to download a folder containing a FAISS index from Google Drive and store it in a local directory in Google Colab. Here's an explanation of each part of the code:

##What is This? 🤔
* folder_id: This is the ID of the folder stored in Google Drive. It uniquely identifies the folder that contains the necessary files, in this case, a FAISS index that will be used for efficient similarity search or document retrieval.
local_index_path: This defines the local directory path in Google Colab where the FAISS index will be downloaded and stored .

* gdown: This is a command-line tool used to download files or folders from Google Drive. In this case, it downloads the entire folder from Google Drive and saves it to the specified local directory.

##Why Do We Need This? ⚙️
* FAISS (Facebook AI Similarity Search) is an efficient library for searching large datasets, especially for similarity searches based on vector embeddings. It is used to quickly retrieve relevant documents or data.
The FAISS index in the folder contains precomputed data for performing these similarity searches. Downloading it to the local directory allows us to use this data for fast retrieval operations without recomputing it every time.

##Key Functions 🔑
* gdown Command:
Downloads the specified Google Drive folder using its folder ID and stores the files in the local directory on Google Colab. This command is crucial for transferring data from Google Drive to Colab's environment for processing.
The -O flag specifies the output path where the folder will be saved locally.



In [ ]:
# Google Drive folder ID for FAISS index
folder_id = "1yn6tvX6pTiq_U3Dfi5kFL8hCCVpmgZI_" # Replace with your actual folder ID
local_index_path = "/content/faiss_index" # Local path to save the index in Colab

# Download FAISS index from Google Drive
print(f"Downloading FAISS index from Google Drive folder ID: {folder_id} to {local_index_path}...")
# Ensure the local directory exists before downloading
os.makedirs(local_index_path, exist_ok=True)
try:
    !gdown --folder {folder_id} -O {local_index_path} --quiet
    print("FAISS index downloaded successfully.")
except Exception as e:
    print(f"Error downloading FAISS index: {e}")
    print("Chatbot might not have context from the index.")

Enter the **API Key** for the AI:

In [ ]:
ai_api_key = os.environ.get("GROQ_API_KEY") # Use GROQ_API_KEY as per standard practice
while not ai_api_key:
    ai_api_key = input("Please enter your GROQ API key: ").strip()
os.environ["GROQ_API_KEY"] = ai_api_key
print("GROQ API key is set.")

###**5:SYSTEM_PROMPT**###
### Setting Up the System Prompt  

🤔 **What Is the `SYSTEM_PROMPT`?**  
The `SYSTEM_PROMPT` is a f‑string that defines your assistant’s persona, scope, and response rules at startup. It tells the model:  
- **Who** it is (“Beta, an expert assistant for the Education Abroad program…”)  
- **What** it’s allowed to talk about (study abroad, visas, scholarships, etc.)  
- **How** it should speak (concise, factual, supportive)  
- **What** to do when it lacks information (“I don’t have enough information…”)

---

🛡️ **Why Do We Need a System Prompt?**  
- **Consistent Persona**  
  Ensures the assistant always introduces itself and frames answers as a CSUSB Education Abroad expert.  
- **Topic Guardrails**  
  Prevents wandering into unrelated or controversial subjects by enforcing explicit “stay on topic” rules.  
- **Quality & Tone Control**  
  Enforces brevity (2–3 sentences), positivity, and factual accuracy.  
- **Failure Mode**  
  Defines a safe fallback (“I don’t have enough information…”) to avoid hallucinations or made‑up details.

---

🔑 **Key Functions in This System Prompt**  
1. **Identity & Expertise**  
   - “You are Beta, an expert assistant for the Education Abroad program…”  
2. **Scope Limitation**  
   - “Only respond to questions related to studying abroad, scholarships, university admissions, visas, or life as an international student.”  
3. **Response Style**  
   - “Keep responses concise: Limit answers to 2–3 sentences.”  
   - “No Negative Responses: Remain factual and avoid discouraging language.”  
4. **Content Safeguards**  
   - “No Controversial Discussions: Do not engage outside approved topics.”  
   - “If the context does not contain enough information… respond with ‘I don’t have enough information…’.”  






In [ ]:

SYSTEM_PROMPT = f"""
You are Beta, an expert assistant for the Education Abroad program of California State University, San Bernardino (CSUSB).
You are designed to help students with all questions related to studying abroad.
You provide detailed, accurate, and helpful information about scholarships, visa processes, university applications, living abroad, cultural adaptation, and academic opportunities worldwide.

Rules & Restrictions:
- **Stay on Topic:** Only respond to questions related to studying abroad, scholarships, university admissions, visas, or life as an international student.
- **No Negative Responses:** Remain factual and avoid discouraging language.
- **Encourage and Inform:** Provide clear, supportive, and correct responses to the approved inquiries.
- **No Controversial Discussions:** Do not engage in topics outside of studying abroad (e.g., politics, religion, or personal debates).
- **Keep Responses Concise:** Limit your answers to 2-3 sentences to ensure brevity and clarity.

Provide a concise and accurate answer based solely on the context below.
If the context does not contain enough information to answer the question, respond with "I don't have enough information to answer this question." Do not generate, assume, or make up any details beyond the given context.
"""

# Global session state for Chatbot
session_state = {
    "cooldownBeginTimestamp": None,
    "messageTimes": [],
    "messages": [],
    "reset": False
}


##**6. Setting Up the Embedding Model and Session State**
###**🤔What is  an Embedding Model?**
An embedding model converts text into numerical vectors that capture the meaning and context of the text. These vectors enable:

* Similarity Matching: Finding similar texts by comparing vector distances.
* Efficient Retrieval: Powering search and recommendation systems.


This model is lightweight and efficient, making it ideal for chatbot applications.



###**Key Functions for Embedding Setup**
* Langchain: Loads a pre-trained transformer model FastEmbedEmbeddings
* FasteEmbedEmbeddings: A lightweight and fast embedding model optimized for semantic search and sentence similarity.

###**Initializing the Reranker Model**
* Reranker: A fine-tuned transformer that re-ranks retrieved search results.
* Model: FlashRank:
  * Takes the initial documents retrieved from the FAISS search and reorders them based on their relevance to the user query using FlashRank. This helps improve the quality of the context provided to the LLM.

In [ ]:
# Initialize the embedding model.
EMBEDDING_MODEL = FastEmbedEmbeddings()
RERANKER = Ranker(max_length=4096)

def rerank_results(question, documents):
    """Rerank search results using FlashRank without comparing Document objects directly."""
    if not documents:
        return []

    # Create pairs for FlashRank
    pairs = [{"id": i, "text": doc.page_content} for i, doc in enumerate(documents)]
    # Get sorted pairs from FlashRank
    results = RERANKER.rerank(RerankRequest(question, pairs))
    # Reorder documents based on sorted indices, taking top 5
    ranked_docs = [result["text"] for result in results[:5]]
    return ranked_docs


##**7.Token Estimation, Random Question Generation, and Input Truncation 🎯**##

##1. Token Estimation 🔢##
The estimate_tokens() function provides a rough estimate of the number of tokens in a given text by counting the words. It assumes that each word roughly corresponds to one token. This helps to understand how much data the AI model will process, especially when dealing with token limits.


##2. Input Truncation ✂️##
The truncate_input() function ensures that the total length of the combined messages doesn’t exceed the set character limit (MAX_AI_INPUT_CHARACTERS). Here's how it works:

* Checks Message Length 📏: Goes through the messages in reverse order and adds them to the list until the total character count exceeds the limit.
* Maintains Relevant Context 📝: Ensures only the most recent and relevant messages are passed to the AI model while preventing it from being overwhelmed by too much input.



In [ ]:
def estimate_tokens(text):
    """Roughly estimate token count based on word count (1 word ≈ 1 token)"""
    return len(text.split())

def truncate_input(messages):
    """Truncate the combined input messages to a maximum of MAX_AI_INPUT_CHARACTERS characters."""
    combined_text = []
    for msg in reversed(messages):
        msg_length = sum(len(part) for part in msg)
        if len(combined_text) + msg_length > MAX_AI_INPUT_CHARACTERS:
            break
        combined_text.append(msg)
    combined_text.reverse()
    return combined_text

###**8.Session Reset and Cooldown Logic 🔄⏳**###
##1. Reset Function 🔄##
The reset() function clears all session-related data to reinitialize the system, ensuring that the chatbot starts fresh for each interaction. The reset process does the following:

##Clears Session Data ❌:##
* Resets cooldown timestamp to None ⏱️.
* Clears message history 📜 (removes previous timestamps and messages).
* Resets the reset flag to False 🚫.
* Confirmation ✅: After resetting, a message is printed to confirm that the session state has been reset:
"Session state has been reset."







##2. Cooldown Logic ⏳##
The canAnswer() function is responsible for checking whether the chatbot is ready to process a new message based on a cooldown mechanism. The logic follows these key steps:

**Cooldown Period Check ⏱️:** If the cooldown timestamp exists, the function checks if the cooldown duration has passed. If enough time has passed, the system allows a new message and resets the cooldown.

**Message Count and Time Gap Check 📊:** If the cooldown hasn't started yet, it verifies if the number of messages is within limits and checks if enough time has passed between messages based on the cooldown check period. If the user exceeds the limit, the system initiates a cooldown.

**Error Notification ⚠️:** If the user exceeds the allowed limit, the system calculates the remaining cooldown time and provides an error message. For example:

"ERROR: You've reached the limit of 5 questions per 2 minutes. Please try again in 1 minute 30 seconds."

Key Variables for Cooldown Logic ⚙️

| **Variable**                   | **Purpose**                                      |
|---------------------------------|--------------------------------------------------|
| `COOLDOWN_DURATION`             | Total cooldown duration (in seconds) ⏳.         |
| `MAX_MESSAGES_BEFORE_COOLDOWN`  | Maximum number of messages before cooldown starts 📈. |
| `COOLDOWN_CHECK_PERIOD`         | Time period (in seconds) between messages for cooldown checks 🕒. |


In [ ]:
def reset():
    session_state["cooldownBeginTimestamp"] = None
    session_state["messageTimes"] = []
    session_state["messages"] = []
    session_state["reset"] = False
    print("\nSession state has been reset.\n")

def canAnswer() -> bool:
    """Check if the user can send a new message based on the cooldown logic.
    """
    currentTimestamp = time.monotonic()
    if session_state["cooldownBeginTimestamp"] is not None:
        if currentTimestamp - session_state["cooldownBeginTimestamp"] >= COOLDOWN_DURATION:
            session_state["cooldownBeginTimestamp"] = None
            return True
    else:
        session_state["messageTimes"] = session_state["messageTimes"][-MAX_MESSAGES_BEFORE_COOLDOWN:] + [currentTimestamp]
        if (
            len(session_state["messageTimes"]) <= MAX_MESSAGES_BEFORE_COOLDOWN or
            session_state["messageTimes"][-1] - session_state["messageTimes"][-MAX_MESSAGES_BEFORE_COOLDOWN - 1] >= COOLDOWN_CHECK_PERIOD
        ):
            return True
        session_state["cooldownBeginTimestamp"] = currentTimestamp

    # Active waiting with countdown updates.
    while currentTimestamp - session_state["cooldownBeginTimestamp"] < COOLDOWN_DURATION:
        cooldownMinutes = int(COOLDOWN_CHECK_PERIOD // 60)
        cooldownSeconds = int(COOLDOWN_CHECK_PERIOD) % 60
        remainingTime = COOLDOWN_DURATION + session_state["cooldownBeginTimestamp"] - currentTimestamp
        remainingMinutes = int(remainingTime // 60)
        remainingSeconds = int(remainingTime) % 60
        print(
            f"WARNING: The app has reached the limit of {MAX_MESSAGES_BEFORE_COOLDOWN} questions per "
            f"{cooldownMinutes} minute{'s' if cooldownMinutes != 1 else ''} {cooldownSeconds} second{'s' if cooldownSeconds != 1 else ''}. "
            f"The app will resume in {remainingMinutes} minute{'s' if remainingMinutes != 1 else ''} "
            f"{remainingSeconds} second{'s' if remainingSeconds != 1 else ''}.", end='\r'
        )
        time.sleep(1)
        currentTimestamp = time.monotonic()
    # Print a newline after the countdown finishes.
    print("")
    return True

In [ ]:
from scrapy.crawler import CrawlerProcess

# Scrapy spider for goabroad.csusb.edu
class GoAbroadSpider(scrapy.Spider):
    name = "goabroad_spider"
    allowed_domains = ["goabroad.csusb.edu"]
    start_urls = ["https://goabroad.csusb.edu/"]

    custom_settings = {
        "DOWNLOAD_DELAY": 1,
        "AUTOTHROTTLE_ENABLED": True,
        "AUTOTHROTTLE_START_DELAY": 1,
        "AUTOTHROTTLE_MAX_DELAY": 3,
    }

    def parse(self, response):
        global URL_HASHES
        global URL_HASHES_PATH
        raw_text_nodes = response.xpath("//body//text()[normalize-space()]").getall()
        joined_text = " ".join(text.strip() for text in raw_text_nodes if text.strip())
        cleaned_text = WHITESPACE_RE.sub(' ', TAG_RE.sub('', joined_text)).strip()
        
        content_hash = hashlib.md5(cleaned_text.encode("utf-8")).hexdigest()

        if URL_HASHES.get(response.url) == content_hash:
            self.logger.info(f"[SKIPPED] No change for {response.url}")
        else:
            URL_HASHES[response.url] = content_hash
            with open(URL_HASHES_PATH, "w") as f:
                json.dump(URL_HASHES, f, indent=2)

            segments = {cleaned_text[i:i + SEGMENT_SIZE].strip() for i in range(0, len(cleaned_text), SEGMENT_SIZE)}
            if session_state.get("vectorstore") is not None:
                session_state["vectorstore"].add_texts(segments, metadatas=[{"url": response.url} for _ in segments])

        internal_links = response.css("a::attr(href)").getall()
        internal_links = list({
            response.urljoin(link) for link in internal_links 
            if urlparse(response.urljoin(link)).hostname and 
            (urlparse(response.urljoin(link)).hostname == "goabroad.csusb.edu" or 
             urlparse(response.urljoin(link)).hostname.endswith(".goabroad.csusb.edu"))
        })

        for link in internal_links:
            yield scrapy.Request(url=link, callback=self.parse)

def runScraper():
    """Run the Scrapy spider to crawl goabroad.csusb.edu."""
    process = CrawlerProcess({
        "USER_AGENT": "Mozilla/5.0 (compatible; CSUSBStudyAbroadBot/1.0)",
        "DOWNLOAD_DELAY": 1,
        "AUTOTHROTTLE_ENABLED": True,
        "AUTOTHROTTLE_START_DELAY": 1,
        "AUTOTHROTTLE_MAX_DELAY": 3,
    })
    process.crawl(GoAbroadSpider)
    process.start()
    if session_state.get("vectorstore") is not None:
        session_state["vectorstore"].save_local(local_index_path)
        print("FAISS index updated and saved.")

def launchAutomaticScraping():
    """Schedule the scraper to run every 24 hours."""
    if DEBUG_MODE:
        return
    scheduler = BackgroundScheduler()
    scheduler.add_job(runScraper, "interval", hours=24)
    scheduler.start()
    print("Automatic scraping scheduled to run every 24 hours.")


##**9. Main Page and Session Management 🔄💬**##

The mainPage() function is a central part of the chatbot's operation. It is responsible for managing the entire chat session from displaying a welcome message to handling the flow of conversation, processing questions, managing the state of the system, and generating responses.

##**What is This? 🤔**##
The mainPage() function orchestrates multiple tasks:

* Session Initialization:
It displays the CSUSB Study Abroad Chatbot welcome message and checks if the session needs to be reset (clearing previous data).
Session Data Handling: It loads previous messages, ensuring the context is maintained for the conversation.

* API and Data Setup:
 Verifies the Groq API key and loads the FAISS index, which is essential for document retrieval.


* Token Management:
It tracks token usage and ensures that the system doesn’t exceed the token limit.
*

##**Why Do We Need This? ⚙️**##
The mainPage() function is essential for the chatbot's flow. It manages:

* Session Control: Ensures that the chatbot operates within limits (e.g., token limits) and keeps track of previous interactions.

* Efficient Use of Resources: By managing token usage and retrieving answers from the FAISS index, the function helps the chatbot process data efficiently, without overloading the system.

##**Key Functions 🔑**##

* Session Reset and Data Loading 🔄

The function checks if the session needs resetting and loads previous messages to maintain context in the conversation.

* Groq API Key and FAISS Index Setup 🔑📚

Verifies the Groq API key to authenticate the API and loads the FAISS index from Google Drive. The FAISS index allows the chatbot to efficiently search for relevant information based on similarity search.


* User can asks the questions through chatbox


* Token Usage Tracking 🧾

It tracks token usage and ensures that the system does not exceed the token limit for each request, which is essential for maintaining performance and preventing errors.



In [ ]:
def main():
    """Runs the command-line chatbot interface in Colab."""
    print("==========================================")
    print("     CSUSB Study Abroad Chatbot (Colab)")
    print("==========================================")
    print("Type 'quit' or 'exit' to end the conversation.")
    print("Type 'reset' to clear the conversation history.")
    print("-" * 40)


    # Load FAISS index
    try:
        print(f"Loading FAISS index from {local_index_path}...")
        # Allow dangerous deserialization is necessary for FAISS in recent versions
        session_state["vectorstore"] = FAISS.load_local(local_index_path, EMBEDDING_MODEL, allow_dangerous_deserialization=True)
        print("FAISS index loaded successfully.")
    except Exception as e:
        print(f"Error loading FAISS index: {e}")
        print("Chatbot will not be able to use context from the index.")
        session_state["vectorstore"] = None # Ensure it's None on failure

    # Initialize Groq API client
    try:
        print("Initializing Groq API client...")
        ai_model = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.1,
            max_tokens=None, # Allow model to determine length within prompt constraints
            timeout=None, # Increased timeout for Colab environment/potential lag
            max_retries=2,
            api_key=os.environ["GROQ_API_KEY"],
        )
        print("Groq API client initialized.")
    except Exception as e:
        print(f"Error initializing ChatGroq: {e}")
        ai_model = None # Ensure it's None on failure
        # If AI model fails to initialize, stop execution as the chatbot can't function
        return

    # --- Chat Loop ---
    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in {"quit", "exit"}:
            print("Exiting chatbot. Goodbye!")
            break
        elif user_input.lower() == "reset":
            reset()
            continue # Start a new loop iteration after reset
        elif not user_input:
            continue # Ignore empty input

        # Check cooldown before processing the input
        if not canAnswer():
            continue # If cooldown is active, the function prints a message and returns False, so we just skip to the next input

        # Add user message to history
        session_state["messages"].append({"role": "human", "content": user_input})
        print("Beta:", end=" ") # Prepare for typing effect or direct print

        responseStartTime = time.monotonic()
        ai_response_content = "I'm sorry, I encountered an error generating a response." # Default error message

        try:
            # 1. Retrieve documents based on user input
            initial_docs = []
            if session_state["vectorstore"]:
                try:
                     print("(Searching index...) ", end="")
                     initial_docs = session_state["vectorstore"].similarity_search(user_input)
                     print("Done.")
                except Exception as e:
                     print(f"Error during similarity search: {e}")
            else:
                print("(Index not loaded, skipping search) ", end="")


            # 2. Rerank documents
            print("(Reranking results...) ", end="")
            ranked_docs_content = rerank_results(user_input, initial_docs)
            print("Done.")

            # 3. Build context string from ranked documents
            # Use a character limit per document snippet similar to the Streamlit code
            context = " ".join([doc[:500] for doc in ranked_docs_content])

            # 4. Prepare messages for the AI model
            # Include system prompt with context, then recent history
            messages_for_ai = [("system", SYSTEM_PROMPT + "\n\nContext:\n" + context)] + \
                              [(m["role"], m["content"]) for m in session_state["messages"][-MAX_HISTORY_TO_USE:]]

            # 5. Truncate messages if total character count is too high
            truncated_messages = truncate_input(messages_for_ai)

            # 6. Invoke the AI model
            if ai_model:
                print("(Generating response...) ", end="")
                # Format messages for ChatGroq's invoke method (expects list of tuples or messages objects)
                # Simple list of tuples (role, content) usually works
                formatted_messages = [{"role": role, "content": content} for role, content in truncated_messages]

                # Handle potential empty formatted_messages after truncation if input was tiny and history was truncated
                if not formatted_messages:
                    ai_response = "Unable to generate a response based on the provided context (input too short or truncated)."
                else:
                     # The actual call to the LLM
                    ai_response = ai_model.invoke(formatted_messages)
                    print("Done.")

                ai_response_content = ai_response.content
            else:
                 print("AI model not initialized.")


        except Exception as e:
            print(f"Error during AI interaction: {e}")
            # Fallback to the default error message

        responseEndTime = time.monotonic()
        responseTime = responseEndTime - responseStartTime

        # Print the AI response
        print(ai_response_content)
        print(f"(Response took {responseTime:.4f} seconds)", flush=True) # Use flush=True to ensure print appears immediately

        # Add AI response to history
        session_state["messages"].append({"role": "ai", "content": ai_response_content})


if __name__ == "__main__":
    main()